## Load the data

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

## Running the agent

In [7]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [8]:
## Define search tool
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [9]:
## Create the runner
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [12]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [13]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='I just found this course late — am I still allowed to join in now?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"late join allowed course join late enrollment registration allowed to join now"}', call_id='call_Lxk09jeToRKMNqzbjV9bG2oi', name='search', type='function_call', id='fc_0ecc36dcf6252bbf006a63440a42c081939f8805c3adfbde8b', caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_Lxk09jeToRKMNqzbjV9bG2oi',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you ne

In [14]:
## Extract the function name and arguments
def extract_tool_calls(messages):
    tool_calls = []
    
    for message in messages:
        if isinstance(message, dict): 
            continue
        
        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })
            
    return tool_calls

In [15]:
tool_calls = extract_tool_calls(result.all_messages)
tool_calls

[{'name': 'search',
  'arguments': '{"query":"late join allowed course join late enrollment registration allowed to join now"}'}]

In [16]:
## Get the origin answer
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [18]:
import json

agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": json.dumps(tool_calls),
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'I just found this course late — am I still allowed to join in now?',
 'answer_agent': 'Yes — you can still join the course now.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': '[{"name": "search", "arguments": "{\\"query\\":\\"late join allowed course join late enrollment registration allowed to join now\\"}"}]',
 'cost': Decimal('0.000987'),
 'document': '74eb249bbf'}

## Processing multiple questions

In [19]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": json.dumps(tool_calls),
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [20]:
## Run it in the small sample parallel
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [23]:
pd.set_option('display.max_colwidth', None)

df_agent = pd.DataFrame(agent_answers)
df_agent.head(2)

,question,answer_agent,answer_orig,tool_calls,cost,document
0,I just found this course late — am I still allowed to join in now?,"Yes — you can still join the course now.\n\nYou don’t need a special confirmation to start; you can begin learning and submitting homework while the submission forms are still open. If you want a certificate, make sure you submit your project while submissions are still being accepted.","Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.","[{""name"": ""search"", ""arguments"": ""{\""query\"":\""late enrollment join course after start allowed allowed to join now late found course\""}""}]",0.00113925,74eb249bbf
1,Is it too late to start the course if I missed the beginning?,"No — you can start whenever you want. The course videos and GitHub materials are available, and you can begin from the beginning even if you missed the start.\n\nA couple of notes:\n- Homework can only be submitted while the form is open, and there are no late submissions once it closes.\n- Missing homework does not affect your certificate; the Capstone project is what matters for passing.\n\nIf you want, I can also point you to the recommended first steps for getting started.","Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.","[{""name"": ""search"", ""arguments"": ""{\""query\"":\""too late to start course missed beginning late start enrolled after course start\""}""}]",0.001392,74eb249bbf


In [24]:
df_agent["cost"].sum()

Decimal('0.06551475')

In [25]:
df_agent.to_csv("data/agent-answers.csv", index=False)

## Judging answer and trajectories

In [26]:
## Define a judge output type with 2 scores
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [27]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [29]:
from evaluation_utils import calc_total_price, llm_structured_retry

## Define judge function
def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [30]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])
agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer matches the ground truth: it says the learner can still join now, and that to receive a certificate they must submit the project while submissions are still being accepted. This preserves the key condition from the original answer.', answer_score='good', trajectory_reasoning='The single search query was relevant to the question about joining the course late. It included the main ideas of late enrollment and whether joining is allowed. One search was sufficient, and there were no redundant tool calls.', trajectory_score='good')

## Running the agent judge

In [31]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [32]:
with ThreadPoolExecutor(max_workers=3) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [34]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [36]:
df_agent_eval = pd.DataFrame(agent_evaluations)
df_agent_eval.head(2)

,question,document,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
0,I just found this course late — am I still allowed to join in now?,74eb249bbf,good,"The agent’s answer matches the ground truth. It correctly says the student can still join the course now, and it includes the key condition that to receive a certificate, the project must be submitted while submissions are still being accepted. The extra detail about no special confirmation and homework is not in the original answer, but it does not contradict the ground truth.",good,"The search query was relevant and included important keywords from the question such as late, join, course, and allowed. Only one search call was made, which is reasonable for a straightforward FAQ-style question. The tool use supported the final answer adequately."
1,Is it too late to start the course if I missed the beginning?,74eb249bbf,bad,"The agent answer does not match the ground truth. The correct answer says it is not too late to start, but for a certificate you must submit the project while submissions are still being accepted. The agent instead adds incorrect details about homework and says missing homework does not affect the certificate, which is not present in the original answer and may be misleading. It does capture the general idea that you can start later, but it misses the key certificate constraint and introduces unsupported information.",good,"The tool call was relevant to the question because it searched for starting late / missing the beginning. However, only one search was used, which is reasonable, and it did not appear duplicated. The query keywords were appropriate enough, though the search did not help the final answer fully."


In [ ]:
calc_total_price(usages)

In [ ]:
print(df_agent_eval["answer_score"].value_counts()) # Check answer score
print(df_agent_eval["trajectory_score"].value_counts()) # Check trajectory score

In [ ]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)